# Imports and Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append(str(Path("..").resolve()))
from src.utils import DATA_RAW, DATA_PROC, REPORTS, get_logger

logger = get_logger("eda")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

FIGSIZE = (12, 5)

# Load Data

In [ ]:
df = pd.read_csv(DATA_RAW / "application_train.csv")
logger.info(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

# Shape and data types

In [ ]:
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"\nTarget distribution:")
print(df["TARGET"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

print(f"\nData types:")
print(df.dtypes.value_counts())

# Checking for class Imbalance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["TARGET"].value_counts()
colors = ["#4C9BE8", "#E8593C"]

bars = ax.bar(["Repaid (0)", "Defaulted (1)"], counts.values, color=colors, width=0.5, edgecolor="white")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", va="bottom", fontsize=11)

ax.set_title("Target Class Distribution — Severe Imbalance", fontsize=13, fontweight="bold")
ax.set_ylabel("Count")
ax.set_ylim(0, counts.max() * 1.18)
sns.despine()
plt.tight_layout()
plt.savefig(REPORTS / "01_class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("Saved: 01_class_imbalance.png")

# Missing value analysis

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing = missing[missing > 0]

print(f"Columns with missing values: {len(missing)} / {df.shape[1]}")
print(f"\nTop 20 by missing %:")
print(missing.head(30).round(2))

# Visualise top 30
fig, ax = plt.subplots(figsize=(12, 7))
missing.head(30).sort_values(ascending=True).plot(kind="barh", ax=ax, color="#4C9BE8", edgecolor="white")
ax.axvline(x=40, color="red", linestyle="--", linewidth=1, label="40% threshold")
ax.set_xlabel("Missing %")
ax.set_title("Top 30 Columns by Missing Data %", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS / "02_missing_values.png", dpi=150, bbox_inches="tight")
plt.show()

# Numerical feature distribution vs Target

In [ ]:
key_numeric = [
    "AMT_CREDIT", "AMT_INCOME_TOTAL", "AMT_ANNUITY",
    "DAYS_BIRTH", "DAYS_EMPLOYED", "EXT_SOURCE_1",
    "EXT_SOURCE_2", "EXT_SOURCE_3"
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(key_numeric):
    for target_val, color, label in [(0, "#4C9BE8", "Repaid"), (1, "#E8593C", "Defaulted")]:
        subset = df[df["TARGET"] == target_val][col].dropna()
        axes[i].hist(subset, bins=50, alpha=0.55, color=color, label=label, density=True)
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].set_yticks([])

fig.suptitle("Key Feature Distributions by Target", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(REPORTS / "03_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# Categorical features vs default rate

In [ ]:
cat_cols = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE",
            "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS"]

fig, axes = plt.subplots(1, len(cat_cols), figsize=(20, 5))

for ax, col in zip(axes, cat_cols):
    rates = df.groupby(col)["TARGET"].mean().sort_values(ascending=False)
    rates.plot(kind="bar", ax=ax, color="#E8593C", alpha=0.8, edgecolor="white")
    ax.set_title(col.replace("NAME_", "").replace("_", " ").title(), fontsize=10)
    ax.set_ylabel("Default Rate")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

fig.suptitle("Default Rate by Categorical Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS / "04_categorical_default_rates.png", dpi=150, bbox_inches="tight")
plt.show()

# EXT_SOURCE correlation (most predictive features)

In [ ]:
ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "TARGET"]
corr = df[ext_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
            ax=ax, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("EXT_SOURCE Correlations with Target", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS / "05_ext_source_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nCorrelation with TARGET:")
print(df[ext_cols].corr()["TARGET"].drop("TARGET").sort_values())

# EDA Summary

In [ ]:
summary = {
    "n_rows": len(df),
    "n_cols": df.shape[1],
    "default_rate": df["TARGET"].mean().round(4),
    "missing_cols_over_40pct": int((missing > 40).sum()),
    "numeric_cols": int((df.dtypes != "object").sum()),
    "categorical_cols": int((df.dtypes == "object").sum()),
}

import json
with open(DATA_PROC / "eda_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("EDA Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")